<a href="https://colab.research.google.com/github/Ricktheus/Artigo-Rag/blob/main/RAG_Hierarquico_NRs_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema RAG Hierárquico para Normas Regulamentadoras (NRs)

Este notebook implementa um sistema de Recuperação Aumentada por Geração (RAG) especializado nas Normas Regulamentadoras brasileiras (NR-05, 06, 10, 11 e 35).

### Instruções de Uso:
Para garantir a funcionalidade em outros ambientes, certifique-se de que seu repositório GitHub contenha:
1. Este arquivo `.ipynb`.
2. Scripts auxiliares: `config.py`, `node_builder.py`, `ingest.py`, `retriever.py`, `query_engine.py`.
3. Pasta `nrs_extraidas/` com os arquivos JSON originais.

In [ ]:
# Instalação silenciosa de dependências
!pip install -q llama-index-core llama-index-vector-stores-qdrant llama-index-embeddings-huggingface llama-index-llms-openai qdrant-client fastembed sentence-transformers torch

### Configuração Automática do Ambiente
Esta célula baixa automaticamente os arquivos de dados necessários (JSONs) para que o sistema funcione sem a necessidade de upload manual.

In [ ]:
import os
import gc

# Configuração do ambiente e download automático de dados
os.makedirs('nrs_extraidas', exist_ok=True)

urls = [
    "https://raw.githubusercontent.com/Ricktheus/Artigo-Rag/main/nrs_extraidas/nr_05_tree.json",
    "https://raw.githubusercontent.com/Ricktheus/Artigo-Rag/main/nrs_extraidas/nr_06_tree.json",
    "https://raw.githubusercontent.com/Ricktheus/Artigo-Rag/main/nrs_extraidas/nr_10_tree.json",
    "https://raw.githubusercontent.com/Ricktheus/Artigo-Rag/main/nrs_extraidas/nr_11_tree.json",
    "https://raw.githubusercontent.com/Ricktheus/Artigo-Rag/main/nrs_extraidas/nr_35_tree.json"
]

print("Validando base de conhecimentos...")
for url in urls:
    dest = os.path.join('nrs_extraidas', url.split('/')[-1])
    if not os.path.exists(dest):
        os.system(f"wget -q {url} -O {dest}")

# Limpeza de memória e locks do banco vetorial
gc.collect()
lock_file = '/content/qdrant_data/.lock'
if os.path.exists(lock_file): os.remove(lock_file)

print("Ambiente configurado com sucesso.")

In [ ]:
# Exemplo de comandos Git caso queira subir via terminal no Colab
# !git config --global user.email "seu-email@exemplo.com"
# !git config --global user.name "Seu Nome"
# !git init
# !git add .
# !git commit -m "Primeiro commit: Sistema RAG Hierarquico NRs"
# !git remote add origin https://seu-token-de-acesso@github.com/usuario/repositorio.git
# !git push -u origin main

**Dica:** Para que os links acima funcionem, você deve subir os arquivos JSON para uma pasta chamada `nrs_extraidas` no seu GitHub e usar o link do botão **'Raw'** do arquivo.

First, let's create the `nrs_extraidas` directory.

In [ ]:
!mkdir -p nrs_extraidas

Now, let's create `config.py`. Please paste the content of your `config.py` file below.

In [ ]:
%%writefile config.py
"""
Configurações Globais do Sistema de RAG Hierárquico para Normas Regulamentadoras (NRs).
Centraliza caminhos, escopo de normas, parâmetros do Qdrant e modelos de Embedding/LLM.
"""

from pathlib import Path

# Diretórios base do projeto
BASE_DIR = Path(__file__).resolve().parent
DATA_DIRS = [BASE_DIR / "nrs_extraidas", BASE_DIR]
STORAGE_DIR = BASE_DIR / "storage"
QDRANT_DATA_DIR = BASE_DIR / "qdrant_data"

# Garantir criação dos diretórios necessários
STORAGE_DIR.mkdir(parents=True, exist_ok=True)
QDRANT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Escopo estrito das 5 Normas Regulamentadoras para o Artigo Acadêmico
SCOPE_NRS = {
    "NR-05": "nr_05_tree.json",
    "NR-06": "nr_06_tree.json",
    "NR-10": "nr_10_tree.json",
    "NR-11": "nr_11_tree.json",
    "NR-35": "nr_35_tree.json",
}

# Configurações do Qdrant Vector Store
COLLECTION_NAME = "normas_regulamentadoras_hierarchical"
ENABLE_HYBRID_SEARCH = True
SPARSE_MODEL_NAME = "Qdrant/bm25"

# Configurações de Embeddings
# Modelo multilíngue de alta performance para a língua portuguesa
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Parâmetros de Recuperação (Retriever)
DEFAULT_TOP_K = 4
ALPHA_HYBRID = 0.5  # 0.5 pondera igualmente busca densa e esparsa

# Caminho para persistência do Docstore (armazenamento de nós completos e relacionamentos)
DOCSTORE_PATH = STORAGE_DIR / "docstore_nodes.json"

Next, let's create `node_builder.py`. Please paste its content.

In [ ]:
%%writefile node_builder.py
"""
Módulo de Construção e Estruturação de Nós (LlamaIndex TextNodes).
Processa os JSONs das Normas Regulamentadoras e reconstrói as relações
hierárquicas (PARENT, CHILD, PREVIOUS, NEXT) e os metadados de referência.
Utiliza UUIDs determinísticos para total compatibilidade com o Qdrant Local.
"""

import json
import uuid
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo
import config


def generate_node_uuid(node_id: str) -> str:
    """
    Gera um UUID v5 determinístico a partir do identificador da norma/item (ex: 'NR10_10.2.4').
    Garante compatibilidade total com os requisitos de ID do Qdrant (UUID ou inteiro).
    """
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, node_id))


def find_json_file(doc_id: str, filename: str) -> Optional[Path]:
    """
    Busca o arquivo JSON da norma nos diretórios configurados.
    """
    for d in config.DATA_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    return None


def derive_parent_chapter_number(chapter_num: str) -> Optional[str]:
    """
    Deriva o identificador do capítulo pai a partir do número do item.
    Exemplos:
        '10.2.1' -> '10.2'
        '10.2.1.1' -> '10.2.1'
        '10.2' -> '10' (se existir) ou None
        'ANEXO I' -> None
    """
    if not chapter_num or "." not in chapter_num:
        return None

    parts = chapter_num.split(".")
    if len(parts) > 1:
        return ".".join(parts[:-1])
    return None


def build_text_for_node(node_dict: dict) -> str:
    """
    Formata o texto principal do nó garantindo que títulos e conteúdos
    sejam representados de forma semanticamente rica para os modelos de embedding.
    """
    title = (node_dict.get("title") or "").strip()
    content = (node_dict.get("content") or "").strip()
    chapter_number = node_dict.get("chapter_number", "")
    doc_id = node_dict.get("document_id", "")

    if title and content:
        return f"[{doc_id} - Item {chapter_number}: {title}]\n{content}"
    elif title and not content:
        return f"[{doc_id} - Seção {chapter_number}: {title}]"
    elif content:
        return f"[{doc_id} - Item {chapter_number}]\n{content}"
    else:
        return f"[{doc_id} - Item {chapter_number}]"


def build_nodes_from_json(json_path: Path) -> List[TextNode]:
    """
    Lê o JSON estrutural de uma NR e cria os objetos TextNode com seus
    metadados e relacionamentos estruturais completos.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    raw_nodes = data.get("nodes", [])
    text_nodes: List[TextNode] = []
    natural_id_lookup: Dict[str, TextNode] = {}

    doc_id_default = data.get("document_id", "NR")

    # 1ª Passagem: Instanciação dos TextNodes com metadados e UUIDs determinísticos
    for raw in raw_nodes:
        natural_node_id = raw.get("node_id")
        node_uuid = generate_node_uuid(natural_node_id)

        doc_id = raw.get("document_id") or doc_id_default
        chapter_number = str(raw.get("chapter_number", ""))
        depth = int(raw.get("depth", 0))
        chapter_path = raw.get("chapter_path", "")
        title = raw.get("title")
        content = raw.get("content", "")
        has_table = bool(raw.get("has_table", False))
        references = raw.get("references", [])

        node_text = build_text_for_node(raw)

        metadata = {
            "node_id": natural_node_id,
            "document_id": doc_id,
            "chapter_number": chapter_number,
            "chapter_path": chapter_path,
            "depth": depth,
            "title": title or "",
            "has_table": has_table,
            "references": references,
        }

        # Criar TextNode do LlamaIndex com UUID válido para o Qdrant
        text_node = TextNode(
            text=node_text,
            id_=node_uuid,
            metadata=metadata,
            excluded_embed_metadata_keys=["references", "has_table"],
            excluded_llm_metadata_keys=["has_table"],
        )

        # Inicializar lista de relacionamentos
        text_node.relationships[NodeRelationship.CHILD] = []

        text_nodes.append(text_node)
        natural_id_lookup[natural_node_id] = text_node

    # 2ª Passagem: Estabelecer relacionamentos estruturais (PARENT, CHILD, PREV, NEXT)
    doc_prefix = text_nodes[0].metadata["document_id"].replace("-", "") if text_nodes else ""

    for i, node in enumerate(text_nodes):
        chapter_num = node.metadata.get("chapter_number", "")

        # Conectar nó anterior (PREVIOUS) e próximo (NEXT) no mesmo documento
        if i > 0:
            node.relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(
                node_id=text_nodes[i - 1].node_id
            )
        if i < len(text_nodes) - 1:
            node.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
                node_id=text_nodes[i + 1].node_id
            )

        # Conectar Pai (PARENT) e Filho (CHILD)
        parent_chap = derive_parent_chapter_number(chapter_num)
        if parent_chap:
            parent_natural_id = f"{doc_prefix}_{parent_chap}"
            if parent_natural_id in natural_id_lookup:
                parent_node = natural_id_lookup[parent_natural_id]
                node.relationships[NodeRelationship.PARENT] = RelatedNodeInfo(
                    node_id=parent_node.node_id
                )

                # Adicionar este nó à lista de filhos do pai
                child_rel = parent_node.relationships.get(NodeRelationship.CHILD)
                if isinstance(child_rel, list):
                    child_rel.append(RelatedNodeInfo(node_id=node.node_id))
                elif child_rel is None:
                    parent_node.relationships[NodeRelationship.CHILD] = [
                        RelatedNodeInfo(node_id=node.node_id)
                    ]

    return text_nodes


def build_all_scope_nodes() -> Tuple[List[TextNode], Dict[str, TextNode]]:
    """
    Processa APENAS as 5 Normas Regulamentadoras definidas no escopo acadêmico:
    NR-05, NR-06, NR-10, NR-11 e NR-35.
    Ignora quaisquer outras NRs presentes no workspace.
    """
    all_nodes: List[TextNode] = []
    global_lookup: Dict[str, TextNode] = {}

    for doc_id, filename in config.SCOPE_NRS.items():
        json_file = find_json_file(doc_id, filename)
        if not json_file:
            print(f"[AVISO] Arquivo {filename} para {doc_id} não encontrado nos diretórios.")
            continue

        print(f"[PROCESSANDO] Lendo nós de {doc_id} a partir de: {json_file.name}")
        nodes = build_nodes_from_json(json_file)
        print(f" -> {len(nodes)} nós estruturados com sucesso para {doc_id}.")

        all_nodes.extend(nodes)
        for n in nodes:
            # Indexa tanto por UUID quanto por ID natural (ex: "NR10_10.2.4")
            global_lookup[n.node_id] = n
            natural_id = n.metadata.get("node_id")
            if natural_id:
                global_lookup[natural_id] = n

    print(f"\n[TOTAL] {len(all_nodes)} nós estruturados no escopo das 5 NRs.")
    return all_nodes, global_lookup


def save_nodes_to_docstore(nodes: List[TextNode], output_path: Path) -> None:
    """
    Serializa os nós e seus metadados/relacionamentos em um arquivo JSON local
    para permitir resolução instantânea no Retriever sem depender de chamadas remotas.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    serializable = []
    for n in nodes:
        node_dict = n.to_dict()
        serializable.append(node_dict)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)
    print(f"[DOCSTORE] {len(nodes)} nós salvos em cache local: {output_path}")


def load_nodes_from_docstore(input_path: Path) -> Dict[str, TextNode]:
    """
    Carrega o mapa de nós previamente estruturados a partir do cache JSON.
    Indexa por UUID e por identificador natural para buscas rápidas.
    """
    if not input_path.exists():
        raise FileNotFoundError(f"Docstore não encontrado em: {input_path}")

    with open(input_path, "r", encoding="utf-8") as f:
        raw_list = json.load(f)

    lookup: Dict[str, TextNode] = {}
    for d in raw_list:
        node = TextNode.from_dict(d)
        lookup[node.node_id] = node
        natural_id = node.metadata.get("node_id")
        if natural_id:
            lookup[natural_id] = node
    return lookup

Now, let's create `ingest.py`. Please paste its content. This file is crucial for fixing the `ModuleNotFoundError`.

In [ ]:
%%writefile ingest.py
"""
Script de Ingestão e Indexação Híbrida (Passos 2 e 3).
Executa:
1. Leitura e estruturação dos nós das 5 NRs do escopo (NR-05, NR-06, NR-10, NR-11, NR-35).
2. Construção de relacionamentos hierárquicos e metadados.
3. Inicialização do Qdrant local (./qdrant_data) com busca híbrida (Dense + Sparse BM25).
4. Geração de embeddings vetoriais e inserção na collection.
5. Persistência do Docstore local para resolução de hierarquia e referências cruzadas.
"""

import sys
import time
from pathlib import Path
from typing import List

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

from qdrant_client import QdrantClient
from llama_index.core import StorageContext, VectorStoreIndex, Settings
from llama_index.core.schema import TextNode
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

import config
from node_builder import build_all_scope_nodes, save_nodes_to_docstore


def setup_embedding_model() -> HuggingFaceEmbedding:
    """
    Configura o modelo de embedding open-source multilíngue.
    """
    print(f"[EMBEDDING] Carregando modelo local: {config.EMBEDDING_MODEL_NAME}...")
    embed_model = HuggingFaceEmbedding(
        model_name=config.EMBEDDING_MODEL_NAME,
        embed_batch_size=32
    )
    Settings.embed_model = embed_model
    return embed_model


def get_qdrant_vector_store(client: QdrantClient) -> QdrantVectorStore:
    """
    Inicializa o QdrantVectorStore com suporte a busca híbrida (Denso + BM25 esparso).
    """
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=config.COLLECTION_NAME,
        enable_hybrid=config.ENABLE_HYBRID_SEARCH,
        fastembed_sparse_model=config.SPARSE_MODEL_NAME if config.ENABLE_HYBRID_SEARCH else None,
    )
    return vector_store


def run_ingestion() -> None:
    """
    Pipeline principal de ingestão e indexação.
    """
    start_time = time.time()
    print("=" * 70)
    print(" INICIANDO PIPELINE DE INGESTÃO E INDEXAÇÃO HÍBRIDA (NRS BRASIL)")
    print("=" * 70)

    # 1. Carregar nós estruturados das 5 NRs
    nodes, lookup = build_all_scope_nodes()
    if not nodes:
        print("[ERRO] Nenhum nó foi carregado. Verifique os arquivos JSON no diretório.")
        sys.exit(1)

    # 2. Salvar cache do Docstore para uso do Retriever
    save_nodes_to_docstore(nodes, config.DOCSTORE_PATH)

    # 3. Configurar Embeddings
    embed_model = setup_embedding_model()

    # 4. Inicializar Qdrant Local em Disco
    print(f"[QDRANT] Conectando ao Qdrant local em: {config.QDRANT_DATA_DIR.resolve()}...")
    client = QdrantClient(path=str(config.QDRANT_DATA_DIR))
    vector_store = get_qdrant_vector_store(client)

    # 5. Criar StorageContext e VectorStoreIndex
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    print(f"[INDEXAÇÃO] Indexando {len(nodes)} nós com embeddings densos e esparsos (BM25)...")
    index = VectorStoreIndex(
        nodes=nodes,
        storage_context=storage_context,
        embed_model=embed_model,
        show_progress=True
    )

    elapsed = time.time() - start_time
    print("-" * 70)
    print(f"[SUCESSO] Ingestão concluída com êxito em {elapsed:.2f} segundos!")
    print(f"[INFO] Coleção '{config.COLLECTION_NAME}' pronta no Qdrant.")
    print(f"[INFO] Dados persistidos em: {config.QDRANT_DATA_DIR}")
    print("=" * 70)


if __name__ == "__main__":
    run_ingestion()

Next up is `retriever.py`. Please provide its content.

In [ ]:
%%writefile retriever.py
"""
Motor de Recuperação Hierárquica e Resolução de Referências Cruzadas (Passo 4).
Implementa o HierarchicalCrossReferenceRetriever:
1. Busca Híbrida Inicial (Dense + Sparse BM25) no Qdrant.
2. Reconstrução de Contexto Hierárquico (expansão automática de nós Pais e Irmãos).
3. Resolução Silenciosa de Referências Cruzadas Inter-Normas (Multi-hop dentro do escopo).
"""

import sys
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

from typing import Dict, List, Optional, Set
from dataclasses import dataclass, field

from qdrant_client import QdrantClient
from llama_index.core import StorageContext, VectorStoreIndex, Settings
from llama_index.core.schema import (
    NodeWithScore,
    TextNode,
    NodeRelationship,
    QueryBundle,
)
from llama_index.core.retrievers import BaseRetriever
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

import config
from node_builder import load_nodes_from_docstore


@dataclass
class ExpandedContextItem:
    """
    Representação estruturada de um nó recuperado e expandido hierarquicamente.
    """
    primary_node: TextNode
    score: float
    parent_node: Optional[TextNode] = None
    sibling_nodes: List[TextNode] = field(default_factory=list)
    cross_references: List[Dict[str, any]] = field(default_factory=list)

    def to_formatted_context(self) -> str:
        """
        Formata o nó primário com sua linhagem de pai, irmãos e referências cruzadas
        em uma string de contexto coesa para o LLM.
        """
        meta = self.primary_node.metadata
        doc_id = meta.get("document_id", "NR")
        chap_num = meta.get("chapter_number", "")
        chap_path = meta.get("chapter_path", "")

        parts = []
        parts.append(f"=== [NORMA: {doc_id} | ITEM: {chap_num} | HIERARQUIA: {chap_path}] ===")

        # 1. Contexto do Nó Pai (se existir e for relevante)
        if self.parent_node:
            p_meta = self.parent_node.metadata
            p_num = p_meta.get("chapter_number", "")
            p_title = p_meta.get("title", "")
            p_text = self.parent_node.text.strip()
            parts.append(f"--- [CONTEXTO GERAL / REGRA PAI: Seção {p_num} {p_title}] ---\n{p_text}")

        # 2. Conteúdo do Nó Primário (Regra Específica)
        parts.append(f"--- [REGRA ESPECÍFICA / DISPOSITIVO CONSULTADO: Item {chap_num}] ---\n{self.primary_node.text.strip()}")

        # 3. Referências Cruzadas Resolvidas
        if self.cross_references:
            parts.append("--- [REFERÊNCIAS CRUZADAS INTER-NORMAS RESOLVIDAS] ---")
            for ref in self.cross_references:
                target_doc = ref.get("target_document", "")
                target_item = ref.get("target_item", "")
                ref_text = ref.get("content", "")
                source_reason = ref.get("source", "")
                parts.append(
                    f"-> [Norma Referenciada: {target_doc} | Dispositivo: {target_item} (Origem: {source_reason})]:\n{ref_text}"
                )

        return "\n\n".join(parts)


class HierarchicalCrossReferenceRetriever(BaseRetriever):
    """
    Retriever customizado que estende o LlamaIndex com raciocínio hierárquico
    e resolução em grafo das referências entre as Normas Regulamentadoras.
    """

    def __init__(
        self,
        index: Optional[VectorStoreIndex] = None,
        docstore: Optional[Dict[str, TextNode]] = None,
        top_k: int = config.DEFAULT_TOP_K,
        expand_parent: bool = True,
        resolve_references: bool = True,
        embed_model: Optional[HuggingFaceEmbedding] = None,
    ):
        super().__init__()
        self.top_k = top_k
        self.expand_parent = expand_parent
        self.resolve_references = resolve_references

        # Inicializar embeddings open-source sem depender de pacotes externos
        if embed_model is not None:
            self.embed_model = embed_model
        else:
            self.embed_model = HuggingFaceEmbedding(model_name=config.EMBEDDING_MODEL_NAME)

        Settings.embed_model = self.embed_model

        # Inicializar docstore local
        if docstore is not None:
            self.docstore = docstore
        else:
            self.docstore = load_nodes_from_docstore(config.DOCSTORE_PATH)

        # Inicializar index / vector store caso não fornecido
        if index is not None:
            self.index = index
        else:
            client = QdrantClient(path=str(config.QDRANT_DATA_DIR))
            vector_store = QdrantVectorStore(
                client=client,
                collection_name=config.COLLECTION_NAME,
                enable_hybrid=config.ENABLE_HYBRID_SEARCH,
                fastembed_sparse_model=config.SPARSE_MODEL_NAME if config.ENABLE_HYBRID_SEARCH else None,
            )
            storage_context = StorageContext.from_defaults(vector_store=vector_store)
            self.index = VectorStoreIndex.from_vector_store(
                vector_store=vector_store,
                embed_model=self.embed_model,
                storage_context=storage_context
            )

        # Base retriever do LlamaIndex com modo híbrido
        self.base_retriever = self.index.as_retriever(
            similarity_top_k=self.top_k,
            vector_store_query_mode="hybrid" if config.ENABLE_HYBRID_SEARCH else "default",
            alpha=config.ALPHA_HYBRID
        )

    def _get_parent_node(self, node: TextNode) -> Optional[TextNode]:
        """
        Recupera o nó pai a partir do relacionamento PARENT no Docstore.
        """
        parent_rel = node.relationships.get(NodeRelationship.PARENT)
        if parent_rel and hasattr(parent_rel, "node_id"):
            parent_id = parent_rel.node_id
            return self.docstore.get(parent_id)
        return None

    def _resolve_cross_references(
        self, primary_node: TextNode, visited_nodes: Set[str]
    ) -> List[Dict[str, any]]:
        """
        Inspeciona metadados de 'references' e busca os nós alvos dentro
        das 5 NRs do escopo acadêmico (NR-05, NR-06, NR-10, NR-11, NR-35).
        """
        resolved: List[Dict[str, any]] = []
        raw_refs = primary_node.metadata.get("references", [])

        for ref in raw_refs:
            target_doc = ref.get("target_document")
            target_node_id = ref.get("target_node")
            source_info = ref.get("source", "")

            # Filtrar estritamente dentro do escopo das 5 NRs
            if target_doc not in config.SCOPE_NRS:
                continue

            # Caso 1: Referência direta com node_id conhecido
            if target_node_id and target_node_id in self.docstore:
                if target_node_id not in visited_nodes:
                    visited_nodes.add(target_node_id)
                    ref_node = self.docstore[target_node_id]
                    resolved.append({
                        "target_document": target_doc,
                        "target_item": ref_node.metadata.get("chapter_number", ""),
                        "content": ref_node.text.strip(),
                        "source": source_info,
                        "node_id": target_node_id
                    })

            # Caso 2: Referência a uma Norma Geral (ex: menção a 'NR-06' ou 'NR-35')
            elif target_doc and target_doc != primary_node.metadata.get("document_id"):
                doc_prefix = target_doc.replace("-", "")
                for candidate_id in [f"{doc_prefix}_{target_doc.split('-')[1]}.1", f"{doc_prefix}_1.1"]:
                    if candidate_id in self.docstore and candidate_id not in visited_nodes:
                        visited_nodes.add(candidate_id)
                        ref_node = self.docstore[candidate_id]
                        resolved.append({
                            "target_document": target_doc,
                            "target_item": ref_node.metadata.get("chapter_number", "Objetivo Geral"),
                            "content": ref_node.text.strip(),
                            "source": source_info,
                            "node_id": candidate_id
                        })
                        break

        return resolved

    def retrieve_expanded(self, query_str: str) -> List[ExpandedContextItem]:
        """
        Executa a busca híbrida e expande contextualmente com nós Pais e Referências Cruzadas.
        """
        # 1. Busca Híbrida Inicial
        initial_nodes_with_score = self.base_retriever.retrieve(query_str)

        expanded_items: List[ExpandedContextItem] = []
        visited_nodes: Set[str] = set()

        for nws in initial_nodes_with_score:
            node_id = nws.node.node_id
            visited_nodes.add(node_id)

            # Obter TextNode completo do Docstore (com todos relacionamentos)
            full_node = self.docstore.get(node_id, nws.node)

            # 2. Reconstrução de Contexto Hierárquico (Nó Pai)
            parent_node = None
            if self.expand_parent:
                parent_node = self._get_parent_node(full_node)

            # 3. Resolução de Referências Cruzadas
            cross_refs = []
            if self.resolve_references:
                cross_refs = self._resolve_cross_references(full_node, visited_nodes)

            expanded_item = ExpandedContextItem(
                primary_node=full_node,
                score=nws.score or 1.0,
                parent_node=parent_node,
                cross_references=cross_refs
            )
            expanded_items.append(expanded_item)

        return expanded_items

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """
        Implementação do método abstrato _retrieve do BaseRetriever do LlamaIndex.
        Retorna nós com o texto completamente enriquecido e expandido.
        """
        expanded_items = self.retrieve_expanded(query_bundle.query_str)

        enriched_nodes_with_score: List[NodeWithScore] = []
        for item in expanded_items:
            formatted_text = item.to_formatted_context()

            enriched_node = TextNode(
                text=formatted_text,
                id_=f"expanded_{item.primary_node.node_id}",
                metadata={
                    **item.primary_node.metadata,
                    "is_hierarchically_expanded": True,
                    "has_parent_context": item.parent_node is not None,
                    "cross_references_count": len(item.cross_references),
                }
            )
            enriched_nodes_with_score.append(
                NodeWithScore(node=enriched_node, score=item.score)
            )

        return enriched_nodes_with_score

Almost there! Let's create `query_engine.py`. Please paste its content.

In [ ]:
%%writefile query_engine.py
"""
Pipeline de Geração de Resposta e Query Engine (Passo 5).
Integra o HierarchicalCrossReferenceRetriever com LLMs (Mock, Local ou OpenAI)
e formata o prompt especializado para análise jurídica e regulatória das NRs.
"""

import os
import sys
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

from typing import Optional, List, Dict, Any
from dataclasses import dataclass

from llama_index.core import Settings, PromptTemplate
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.schema import NodeWithScore
from llama_index.core.base.response.schema import Response
from llama_index.core.llms import LLM
from llama_index.core.llms.mock import MockLLM

from retriever import HierarchicalCrossReferenceRetriever, ExpandedContextItem
import config


# Prompt especializado para o artigo acadêmico
NR_SYSTEM_PROMPT = """Você é um Engenheiro de Segurança do Trabalho e Jurista especialista em Normas Regulamentadoras do Brasil (NR-05, NR-06, NR-10, NR-11 e NR-35).

Sua tarefa é responder à pergunta do usuário utilizando ESTRITAMENTE o contexto hierárquico e as referências cruzadas fornecidas abaixo.

Instruções para Resposta Acadêmica e Técnica:
1. **Estrutura Hierárquica**: Diferencie a Regra Geral (Seção/Capítulo Pai) das Exigências Específicas e Exceções (Itens/Subitens Filhos).
2. **Citação Obrigatória**: Cite expressamente as Normas (ex: NR-10, NR-06) e a numeração exata de cada item (ex: 10.2.4, 6.5.1).
3. **Referências Cruzadas**: Caso o contexto contenha referências cruzadas entre normas (ex: uma exigência de NR-10 que demanda EPIs da NR-06 ou trabalho em altura da NR-35), explique a integração entre elas.
4. **Fidelidade Normativa**: Não invente regras; atenha-se ao texto legal extraído.

---------------------
CONTEXTO RECUPERADO (HIERARQUIA E REFERÊNCIAS CRUZADAS):
{context_str}
---------------------

PERGUNTA: {query_str}

RESPOSTA TÉCNICA FUNDAMENTADA:"""

PROMPT_TEMPLATE = PromptTemplate(NR_SYSTEM_PROMPT)


class SimpleAcademicSynthesizer:
    """
    Sintetizador acadêmico local para demonstração imediata sem custos de API externa.
    Realiza a compilação estruturada dos achados hierárquicos e normativos.
    """
    def synthesize(self, query_str: str, expanded_items: List[ExpandedContextItem]) -> str:
        if not expanded_items:
            return "Nenhuma norma regulamentadora relevante foi encontrada no escopo das 5 NRs para esta consulta."

        consulted_nrs = set()
        citations = []
        cross_refs_found = []

        lines = []
        lines.append(f"### Parecer Tecnico-Regulatorio sobre: \"{query_str}\"\n")
        lines.append("**1. Fundamentacao Normativa e Hierarquia:**")

        for idx, item in enumerate(expanded_items, 1):
            meta = item.primary_node.metadata
            doc = meta.get("document_id", "NR")
            chap = meta.get("chapter_number", "")
            path = meta.get("chapter_path", "")
            consulted_nrs.add(doc)
            citations.append(f"{doc} (Item {chap})")

            parent_info = ""
            if item.parent_node:
                p_meta = item.parent_node.metadata
                p_num = p_meta.get("chapter_number", "")
                p_title = p_meta.get("title", "")
                parent_info = f" (sob a regra geral da Secao {p_num}: *{p_title}*)" if p_title else f" (sob a Secao {p_num})"

            lines.append(f"- **Dispositivo Principal {idx} [{doc} - Item {chap}]:**{parent_info}")
            lines.append(f"  *Hierarquia:* `{path}`")
            lines.append(f"  *Texto Normativo:* \"{item.primary_node.text.strip()}\"\n")

            if item.cross_references:
                for ref in item.cross_references:
                    target_doc = ref.get("target_document")
                    target_item = ref.get("target_item")
                    ref_content = ref.get("content")
                    consulted_nrs.add(target_doc)
                    cross_refs_found.append(f"{doc} -> {target_doc} ({target_item})")
                    lines.append(f"  ↳ **Articulacao com {target_doc} (Item {target_item}):**")
                    lines.append(f"    *Exigencia Complementar:* \"{ref_content}\"\n")

        lines.append("**2. Conclusao e Aplicacao Pratica:**")
        lines.append(
            f"A analise integrada das normas ({', '.join(sorted(consulted_nrs))}) demonstra a necessidade de cumprimento "
            f"conjunto tanto das regras gerais de gestao de seguranca quanto dos dispositivos especificos citados ({', '.join(citations)})."
        )
        if cross_refs_found:
            lines.append(f"Destaca-se a interconexao regulamentar identificada: {'; '.join(cross_refs_found)}.")

        return "\n".join(lines)


class NRHierarchicalQueryEngine(CustomQueryEngine):
    """
    Query Engine customizado que combina o HierarchicalCrossReferenceRetriever
    com o mecanismo de síntese e formatação acadêmica.
    """
    retriever: HierarchicalCrossReferenceRetriever
    llm: Optional[LLM] = None
    use_academic_synthesizer: bool = True

    def __init__(
        self,
        retriever: HierarchicalCrossReferenceRetriever,
        llm: Optional[LLM] = None,
        use_academic_synthesizer: bool = False,
    ):
        super().__init__(
            retriever=retriever,
            llm=llm,
            use_academic_synthesizer=use_academic_synthesizer
        )

    def custom_query(self, query_str: str) -> Response:
        """
        Executa o pipeline completo:
        1. Recuperação Híbrida + Expansão Hierárquica + Resolução de Referências
        2. Injeção no Prompt do LLM ou Sintetizador Acadêmico
        3. Geração da Resposta Estruturada
        """
        # Obter itens expandidos para inspeção detalhada
        expanded_items = self.retriever.retrieve_expanded(query_str)
        nodes_with_score = self.retriever.retrieve(query_str)

        # Montar contexto concatenado
        context_parts = [item.to_formatted_context() for item in expanded_items]
        full_context_str = "\n\n" + ("=" * 50) + "\n\n".join(context_parts)

        # Se houver LLM real configurado (ex: OpenAI / Groq / Ollama)
        if self.llm and not isinstance(self.llm, MockLLM) and not self.use_academic_synthesizer:
            formatted_prompt = PROMPT_TEMPLATE.format(
                context_str=full_context_str,
                query_str=query_str
            )
            llm_output = self.llm.complete(formatted_prompt)
            response_text = str(llm_output)
        else:
            # Sintetizador Estruturado Acadêmico
            synthesizer = SimpleAcademicSynthesizer()
            response_text = synthesizer.synthesize(query_str, expanded_items)

        return Response(
            response=response_text,
            source_nodes=nodes_with_score,
            metadata={
                "expanded_items_count": len(expanded_items),
                "consulted_nrs": list(set(
                    item.primary_node.metadata.get("document_id") for item in expanded_items
                )),
                "raw_context": full_context_str
            }
        )


def build_query_engine(
    retriever: Optional[HierarchicalCrossReferenceRetriever] = None,
    openai_api_key: Optional[str] = None,
    use_mock: bool = False
) -> NRHierarchicalQueryEngine:
    """
    Factory function para instanciar o Query Engine configurado.
    """
    if retriever is None:
        retriever = HierarchicalCrossReferenceRetriever()

    llm = None
    api_key = openai_api_key or os.environ.get("OPENAI_API_KEY")

    if api_key and not use_mock:
        try:
            from llama_index.llms.openai import OpenAI
            llm = OpenAI(model="gpt-4o-mini", api_key=api_key)
            use_academic_synthesizer = False
            print("[LLM] Utilizando OpenAI (gpt-4o-mini) para geracao.")
        except Exception as e:
            print(f"[LLM] OpenAI indisponivel ({e}). Usando sintetizador academico local.")
            use_academic_synthesizer = True
    else:
        # Modo local / mock gratuito sem dependência de API
        use_academic_synthesizer = True
        llm = MockLLM()
        print("[LLM] Modo local/sintetizador academico ativo (custo zero, 100% offline).")

    return NRHierarchicalQueryEngine(
        retriever=retriever,
        llm=llm,
        use_academic_synthesizer=use_academic_synthesizer
    )

Finally, let's create `main.py`. Please provide its content.

In [ ]:
%%writefile main.py
"""
Script Principal e Interface de Demonstração (main.py).
Permite executar testes automatizados com casos reais das 5 NRs
ou iniciar um loop interativo de perguntas e respostas.
"""

import sys
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

import argparse
from typing import List

from query_engine import build_query_engine
from retriever import HierarchicalCrossReferenceRetriever


BENCHMARK_QUERIES = [
    {
        "id": "TEST_01",
        "description": "Exigências de Prontuário Elétrico e Integração com EPIs (NR-10 e NR-06)",
        "query": "Quais são os estabelecimentos obrigados a manter o Prontuário de Instalações Elétricas e o que ele deve conter quanto a equipamentos de proteção?"
    },
    {
        "id": "TEST_02",
        "description": "Definição de Trabalho em Altura e Proteção Contra Quedas (NR-35 e NR-06)",
        "query": "A partir de qual altura uma atividade é considerada trabalho em altura e quais as responsabilidades da organização e EPIs contra queda?"
    },
    {
        "id": "TEST_03",
        "description": "Estrutura e Atribuições da CIPA (NR-05)",
        "query": "Quais são as principais atribuições da CIPA e como deve ser conduzido o processo eleitoral?"
    },
    {
        "id": "TEST_04",
        "description": "Operação de Equipamentos de Movimentação de Cargas (NR-11)",
        "query": "Quais os requisitos de segurança para poços de elevadores, guindastes e empilhadeiras na movimentação de materiais?"
    },
    {
        "id": "TEST_05",
        "description": "Responsabilidades do Empregador e Empregado sobre EPI (NR-06)",
        "query": "Quais são as responsabilidades da organização e do trabalhador quanto ao fornecimento, uso e guarda do EPI?"
    }
]


def run_benchmark_tests(engine) -> None:
    """
    Executa a bateria de testes conceituais para validação acadêmica do RAG Hierárquico.
    """
    print("\n" + "=" * 80)
    print(" INICIANDO BATERIA DE TESTES DO RAG HIERARQUICO (5 NRs EM ESCOPO)")
    print("=" * 80)

    for case in BENCHMARK_QUERIES:
        print("\n" + "#" * 80)
        print(f" CASO DE TESTE [{case['id']}]: {case['description']}")
        print(f" [PERGUNTA]: \"{case['query']}\"")
        print("#" * 80)

        response = engine.custom_query(case["query"])

        print("\n[RESULTADO DA GERACAO]:\n")
        print(response.response)

        print("\n" + "-" * 35 + " [DIAGNOSTICO DE RECUPERACAO] " + "-" * 35)
        print(f" * Total de Nos Expandidos: {response.metadata.get('expanded_items_count')}")
        print(f" * Normas Regulamentadoras Consultadas: {response.metadata.get('consulted_nrs')}")
        print("-" * 80)


def interactive_session(engine) -> None:
    """
    Inicia uma sessão interativa de perguntas no terminal.
    """
    print("\n" + "=" * 80)
    print(" MODO INTERATIVO - RAG HIERARQUICO DE NORMAS REGULAMENTADORAS")
    print(" Escopo: NR-05, NR-06, NR-10, NR-11, NR-35")
    print(" Digite sua pergunta ou 'sair' para encerrar.")
    print("=" * 80 + "\n")

    while True:
        try:
            query = input("\n[Pergunta]> ").strip()
            if not query:
                continue
            if query.lower() in ("sair", "exit", "quit"):
                print("Encerrando sessao interativa. Ate logo!")
                break

            response = engine.custom_query(query)
            print("\n" + "=" * 50 + " RESPOSTA " + "=" * 50 + "\n")
            print(response.response)
            print("\n" + "=" * 108 + "\n")
            print(f"[INFO] NRs Consultadas: {response.metadata.get('consulted_nrs')}")

        except KeyboardInterrupt:
            print("\nEncerrando...")
            break
        except Exception as e:
            print(f"\n[ERRO] Ocorreu uma excecao durante o processamento: {e}")


def main():
    parser = argparse.ArgumentParser(
        description="Sistema de RAG Hierarquico para Normas Regulamentadoras (NRs)"
    )
    parser.add_argument(
        "--test",
        action="store_true",
        help="Executa a bateria de testes automatizados com casos de estudo das 5 NRs."
    )
    parser.add_argument(
        "--query",
        type=str,
        default=None,
        help="Executa uma consulta direta passada via linha de comando."
    )
    parser.add_argument(
        "--top-k",
        type=int,
        default=4,
        help="Numero de nos principais a recuperar (padrao: 4)."
    )

    args = parser.parse_args()

    print("[INICIALIZANDO] Carregando motor de recuperacao hierarquico...")
    retriever = HierarchicalCrossReferenceRetriever(top_k=args.top_k)
    engine = build_query_engine(retriever=retriever)

    if args.test:
        run_benchmark_tests(engine)
    elif args.query:
        response = engine.custom_query(args.query)
        print("\n" + "=" * 60)
        print(response.response)
        print("=" * 60)
        print(f"\nNRs Consultadas: {response.metadata.get('consulted_nrs')}")
    else:
        # Padrão: Modo interativo
        interactive_session(engine)


if __name__ == "__main__":
    main()

In [ ]:

from ingest import run_ingestion

# Executa a ingestão e indexação híbrida
run_ingestion()

In [ ]:
import importlib
import ingest
import node_builder
import config

# Recarrega para garantir que as novas NRs sejam detectadas
importlib.reload(config)
importlib.reload(node_builder)
importlib.reload(ingest)

print("Modulos prontos. Iniciando processamento das NRs...")

# Executa a ingestao dos arquivos que voce acabou de subir
ingest.run_ingestion()

In [ ]:
from google.colab import files
import os

# Garante que o diretório existe
os.makedirs('nrs_extraidas', exist_ok=True)

print("Por favor, selecione os arquivos JSON das NRs (nr_05_tree.json, etc.):")
uploaded = files.upload()

# Move os arquivos para a pasta correta
for filename in uploaded.keys():
    os.rename(filename, os.path.join('nrs_extraidas', filename))

print("✅ Arquivos movidos para nrs_extraidas/. Agora você pode rodar a célula de ingestão novamente.")

```markdown
### 🏗️ 1. Ingestão e Indexação
Esta etapa processa os JSONs estruturados, cria os nós com relacionamentos hierárquicos e gera os embeddings no banco vetorial Qdrant.
```

In [ ]:
import ingest
import importlib
importlib.reload(ingest)

# Execução do pipeline de indexação
ingest.run_ingestion()

In [ ]:
import os
import config
import gc
from ingest import run_ingestion

# Tenta forçar a limpeza de instâncias anteriores do Qdrant para liberar o lock
gc.collect()

# Garante que as pastas de armazenamento existam
os.makedirs(config.STORAGE_DIR, exist_ok=True)
os.makedirs(config.QDRANT_DATA_DIR, exist_ok=True)

print("Reexecutando a ingestão para gerar o Docstore...")
try:
    run_ingestion()
except RuntimeError as e:
    if "already accessed" in str(e):
        print("⚠️ O banco de dados está travado. Reiniciando o Kernel ou limpando variáveis...")
        # Em alguns casos no Colab, se o lock persistir, é necessário deletar a variável do cliente antigo
        # ou reiniciar a sessão. Tentaremos prosseguir se o docstore já existir.
    else:
        raise e

if os.path.exists(config.DOCSTORE_PATH):
    print(f"✅ Sucesso! Docstore disponível em: {config.DOCSTORE_PATH}")
else:
    print("❌ O arquivo docstore_nodes.json não foi encontrado.")

In [ ]:
import gc
import os
import shutil

# 1. Tenta forcar a limpeza de memoria
gc.collect()

# 2. Se o erro persistir, a solucao mais segura no Colab e remover o arquivo de lock manual
lock_file = "/content/qdrant_data/.lock"
if os.path.exists(lock_file):
    try:
        os.remove(lock_file)
        print("Arquivo de lock removido com sucesso.")
    except Exception as e:
        print(f"Nao foi possivel remover o lock: {e}")

print("Pronto! Agora tente rodar a celula de inicializacao do Retriever novamente.")

```markdown
### 🔍 2. Inicialização do Motor de RAG
Configuração do Retriever Hierárquico e do Query Engine para processamento de consultas técnicas.
```

In [ ]:
from retriever import HierarchicalCrossReferenceRetriever
from query_engine import build_query_engine

# Inicialização do motor RAG
retriever = HierarchicalCrossReferenceRetriever(top_k=4)
engine = build_query_engine(retriever=retriever)

print("Sistema RAG Hierárquico pronto para consultas.")

```markdown
### 📝 3. Execução de Pareceres Técnicos
Exemplo de consulta ao sistema para geração de parecer fundamentado.
```

In [ ]:
pergunta = "Quais estabelecimentos devem constituir o Prontuario de Instalacoes Eletricas na NR-10 e quais EPIs sao exigidos?"
resposta = engine.custom_query(pergunta)

print(resposta.response)
print("\nConsultadas:", resposta.metadata.get("consulted_nrs"))

After executing all these cells with the correct content, you should be able to run the `M5DYLkV3tDrU` cell without the `ModuleNotFoundError`.

In [ ]:
import os
import gc
from retriever import HierarchicalCrossReferenceRetriever
from query_engine import build_query_engine
import config

# 1. Forcar limpeza de memoria e referencias ao Qdrant
gc.collect()

# 2. Remover manualmente o arquivo de lock se ele existir
lock_file = os.path.join(config.QDRANT_DATA_DIR, ".lock")
if os.path.exists(lock_file):
    try:
        os.remove(lock_file)
        print("Arquivo de lock removido.")
    except Exception as e:
        print(f"Aviso ao remover lock: {e}")

print("Carregando motor de busca hierarquico...")

try:
    # Inicializa o retriever hierarquico com top_k=4
    retriever = HierarchicalCrossReferenceRetriever(top_k=4)
    # Inicializa o Query Engine
    engine = build_query_engine(retriever=retriever)
    print("Motor de RAG Hierarquico pronto para consultas!")
except Exception as e:
    print(f"Erro ao inicializar: {e}")

In [ ]:
from main import run_benchmark_tests

# Executa os 5 cenários de teste cobrindo todas as NRs e referências cruzadas
run_benchmark_tests(engine)

In [ ]:
pergunta = "Quais estabelecimentos devem constituir o Prontuário de Instalações Elétricas na NR-10 e quais EPIs são exigidos?"

resposta = engine.custom_query(pergunta)
print(resposta.response)
print("\n🔍 NRs Consultadas:", resposta.metadata.get("consulted_nrs"))



In [ ]:
# Teste uma consulta de integração entre NRs
pergunta_customizada = "Quais os requisitos para seleção de EPIs em atividades de trabalho em altura conforme a NR-35 e NR-06?"

resposta = engine.custom_query(pergunta_customizada)
print(f"--- RESULTADO DA CONSULTA ---\n")
print(resposta.response)
print(f"\n🔍 NRs Relacionadas Identificadas: {resposta.metadata.get('consulted_nrs')}")

### 🧹 Limpeza de Ambiente (Opcional)
Execute esta célula se desejar remover os arquivos de log e o banco de dados local antes de baixar o notebook para backup manual.

In [ ]:
# Remove arquivos temporários de banco de dados e armazenamento local para exportação limpa
import shutil
import os

def limpar_projeto():
    pastas = ['qdrant_data', 'storage', '__pycache__']
    for pasta in pastas:
        if os.path.exists(pasta):
            shutil.rmtree(pasta)
            print(f"Removido: {pasta}")

    if os.path.exists('nrs_extraidas/.lock'):
        os.remove('nrs_extraidas/.lock')

# limpar_projeto() # Descomente para executar a limpeza física dos arquivos

In [ ]:
# Teste uma consulta de integração entre NRs
pergunta_customizada = "Em quais situações uma empresa está dispensada de constituir CIPA e como deve ser feita a designação do responsável?"

resposta = engine.custom_query(pergunta_customizada)
print(f"--- RESULTADO DA CONSULTA ---\n")
print(resposta.response)
print(f"\n🔍 NRs Relacionadas Identificadas: {resposta.metadata.get('consulted_nrs')}")